In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Cargar 6 temporadas
temporadas = ["2021", "2122", "2223", "2324", "2425", "2526"]
base_url = "https://www.football-data.co.uk/mmz4281/{}/E0.csv"

dfs = []
for t in temporadas:
    df_temp = pd.read_csv(base_url.format(t))
    df_temp["season"] = t
    dfs.append(df_temp)

df_raw = pd.concat(dfs, ignore_index=True)

# Columnas que necesitamos
cols = ['season', 'Date', 'HomeTeam', 'AwayTeam','FTHG', 'FTAG', 'FTR', 'HS', 'AS', 'HST', 'AST', 'HC', 'AC']

df = df_raw[cols].dropna(subset=['FTR'])
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True)
df = df.sort_values('Date').reset_index(drop=True)

print(f"Total partidos: {len(df)}")
print(f"Rango fechas: {df['Date'].min().date()} → {df['Date'].max().date()}")
print(f"\nDistribución resultados:")
print(df['FTR'].value_counts())

Total partidos: 2209
Rango fechas: 2020-09-12 → 2026-03-22

Distribución resultados:
FTR
H    950
A    742
D    517
Name: count, dtype: int64


In [2]:
def get_team_stats(team, date, df, n=5):
    """
    Calcula stats de los últimos n partidos de un equipo
    antes de una fecha determinada.
    """
    # Partidos como local y visitante antes de la fecha
    home = df[(df['HomeTeam'] == team) & (df['Date'] < date)].tail(n)
    away = df[(df['AwayTeam'] == team) & (df['Date'] < date)].tail(n)
    
    partidos = pd.concat([home, away]).sort_values('Date').tail(n)
    
    if len(partidos) < 3:  # mínimo 3 partidos para stats confiables
        return None
    
    stats = {}
    gf, gc, sot_f, sot_c, wins, draws = 0, 0, 0, 0, 0, 0
    
    for _, row in partidos.iterrows():
        if row['HomeTeam'] == team:
            gf += row['FTHG']; gc += row['FTAG']
            sot_f += row['HST']; sot_c += row['AST']
            if row['FTR'] == 'H': wins += 1
            elif row['FTR'] == 'D': draws += 1
        else:
            gf += row['FTAG']; gc += row['FTHG']
            sot_f += row['AST']; sot_c += row['HST']
            if row['FTR'] == 'A': wins += 1
            elif row['FTR'] == 'D': draws += 1
    
    m = len(partidos)
    stats['gf_pg']      = gf / m
    stats['gc_pg']      = gc / m
    stats['dif_goles']  = (gf - gc) / m
    stats['sot_f_pg']   = sot_f / m
    stats['sot_c_pg']   = sot_c / m
    stats['win_rate']   = wins / m
    stats['draw_rate']  = draws / m
    
    return stats

print("Función definida correctamente")
print("Probando con Arsenal al 2026-01-01:")
test = get_team_stats('Arsenal', pd.Timestamp('2026-01-01'), df)
print(test)

Función definida correctamente
Probando con Arsenal al 2026-01-01:
{'gf_pg': 2.0, 'gc_pg': 1.0, 'dif_goles': 1.0, 'sot_f_pg': 5.2, 'sot_c_pg': 3.0, 'win_rate': 0.8, 'draw_rate': 0.0}


In [3]:
def show_team_stats(team, date_str):
    stats = get_team_stats(team, pd.Timestamp(date_str), df)
    df_stats = pd.DataFrame([stats], index=[team])
    df_stats.columns = ['Goles favor/PG', 'Goles contra/PG', 'Dif goles', 
                        'SOT favor/PG', 'SOT contra/PG', 'Win rate', 'Draw rate']
    df_stats = df_stats.round(2)
    return df_stats

# Prueba con varios equipos
equipos_test = ['Arsenal', 'Man City', 'Liverpool', 'Chelsea', 'Tottenham']
fecha = '2026-03-01'

pd.concat([show_team_stats(eq, fecha) for eq in equipos_test])

,Goles favor/PG,Goles contra/PG,Dif goles,SOT favor/PG,SOT contra/PG,Win rate,Draw rate
Arsenal,2.8,0.8,2.0,5.2,2.8,0.6,0.4
Man City,2.0,0.8,1.2,5.4,4.4,0.8,0.2
Liverpool,2.4,1.0,1.4,5.0,3.4,0.8,0.0
Chelsea,2.4,1.4,1.0,4.8,4.4,0.6,0.4
Tottenham,1.2,2.4,-1.2,5.0,5.8,0.0,0.4


In [4]:
def show_matchup(home, away, date_str):
    home_s = get_team_stats(home, pd.Timestamp(date_str), df)
    away_s = get_team_stats(away, pd.Timestamp(date_str), df)
    
    df_match = pd.DataFrame([home_s, away_s], 
                             index=[f'{home} (local)', f'{away} (visitante)'])
    df_match.columns = ['Goles favor/PG', 'Goles contra/PG', 'Dif goles',
                        'SOT favor/PG', 'SOT contra/PG', 'Win rate', 'Draw rate']
    return df_match.round(2)

show_matchup('Arsenal', 'Chelsea', '2026-03-01')

,Goles favor/PG,Goles contra/PG,Dif goles,SOT favor/PG,SOT contra/PG,Win rate,Draw rate
Arsenal (local),2.8,0.8,2.0,5.2,2.8,0.6,0.4
Chelsea (visitante),2.4,1.4,1.0,4.8,4.4,0.6,0.4


In [5]:
rows = []

for idx, partido in df.iterrows():
    home_team = partido['HomeTeam']
    away_team = partido['AwayTeam']
    date      = partido['Date']
    season    = partido['season']
    
    # Stats de ambos equipos antes de este partido
    home_stats = get_team_stats(home_team, date, df)
    away_stats = get_team_stats(away_team, date, df)
    
    # Saltar si alguno no tiene suficiente historia
    if home_stats is None or away_stats is None:
        continue
    
    row = {
        'date':          date,
        'season':        season,
        'home_team':     home_team,
        'away_team':     away_team,
        'resultado':     partido['FTR'],
        # Stats equipo local
        'h_gf_pg':       home_stats['gf_pg'],
        'h_gc_pg':       home_stats['gc_pg'],
        'h_dif_goles':   home_stats['dif_goles'],
        'h_sot_f_pg':    home_stats['sot_f_pg'],
        'h_sot_c_pg':    home_stats['sot_c_pg'],
        'h_win_rate':    home_stats['win_rate'],
        'h_draw_rate':   home_stats['draw_rate'],
        # Stats equipo visitante
        'a_gf_pg':       away_stats['gf_pg'],
        'a_gc_pg':       away_stats['gc_pg'],
        'a_dif_goles':   away_stats['dif_goles'],
        'a_sot_f_pg':    away_stats['sot_f_pg'],
        'a_sot_c_pg':    away_stats['sot_c_pg'],
        'a_win_rate':    away_stats['win_rate'],
        'a_draw_rate':   away_stats['draw_rate'],
        # Features diferenciales
        'dif_gf':        home_stats['gf_pg']    - away_stats['gf_pg'],
        'dif_gc':        home_stats['gc_pg']    - away_stats['gc_pg'],
        'dif_wr':        home_stats['win_rate'] - away_stats['win_rate'],
        'dif_sot':       home_stats['sot_f_pg'] - away_stats['sot_f_pg'],
    }
    rows.append(row)

df_model = pd.DataFrame(rows)

print(f"Partidos en el dataset final: {len(df_model)}")
print(f"Features: {len(df_model.columns) - 5}")  # sin date, season, teams, resultado
print(f"\nDistribución resultados:")
print(df_model['resultado'].value_counts())
print(f"\nTemporadas:")
print(df_model['season'].value_counts().sort_index())

Partidos en el dataset final: 2153
Features: 18

Distribución resultados:
resultado
H    924
A    719
D    510
Name: count, dtype: int64

Temporadas:
season
2021    348
2122    371
2223    374
2324    377
2425    377
2526    306
Name: count, dtype: int64


In [6]:
from sklearn.preprocessing import LabelEncoder

# Split temporal — train: primeras 5 temporadas, test: 2526
train = df_model[df_model['season'] != '2526'].copy()
test  = df_model[df_model['season'] == '2526'].copy()

# Features y target
features = [
    'h_gf_pg', 'h_gc_pg', 'h_dif_goles', 'h_sot_f_pg', 'h_sot_c_pg',
    'h_win_rate', 'h_draw_rate',
    'a_gf_pg', 'a_gc_pg', 'a_dif_goles', 'a_sot_f_pg', 'a_sot_c_pg',
    'a_win_rate', 'a_draw_rate',
    'dif_gf', 'dif_gc', 'dif_wr', 'dif_sot'
]

X_train = train[features]
X_test  = test[features]

# Encodear target: A=0, D=1, H=2
le = LabelEncoder()
y_train = le.fit_transform(train['resultado'])
y_test  = le.transform(test['resultado'])

print(f"Train: {len(X_train)} partidos ({train['season'].min()} → {train['season'].max()})")
print(f"Test:  {len(X_test)} partidos (temporada 2526)")
print(f"\nClases: {le.classes_}")
print(f"\nDistribución train:")
print(train['resultado'].value_counts())
print(f"\nDistribución test:")
print(test['resultado'].value_counts())

Train: 1847 partidos (2021 → 2425)
Test:  306 partidos (temporada 2526)

Clases: ['A' 'D' 'H']

Distribución train:
resultado
H    798
A    623
D    426
Name: count, dtype: int64

Distribución test:
resultado
H    126
A     96
D     84
Name: count, dtype: int64


In [7]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, log_loss
from sklearn.calibration import CalibratedClassifierCV

# Entrenar XGBoost
xgb = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

xgb.fit(X_train, y_train)

# Calibrar probabilidades
model_cal = CalibratedClassifierCV(xgb, cv=5, method='isotonic')
model_cal.fit(X_train, y_train)

# Evaluar
y_pred       = model_cal.predict(X_test)
y_pred_proba = model_cal.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
logloss  = log_loss(y_test, y_pred_proba)

print(f"Accuracy:  {accuracy:.1%}")
print(f"Log Loss:  {logloss:.4f}")

# Distribución de predicciones
pred_labels = le.inverse_transform(y_pred)
print(f"\nPredicciones del modelo:")
print(pd.Series(pred_labels).value_counts())

# Comparar contra baseline (siempre predecir H)
baseline = accuracy_score(y_test, [2]*len(y_test))
print(f"\nBaseline (siempre H): {baseline:.1%}")
print(f"Mejora vs baseline:   {(accuracy - baseline):.1%}")

Accuracy:  43.1%
Log Loss:  1.0808

Predicciones del modelo:
H    209
A     95
D      2
Name: count, dtype: int64

Baseline (siempre H): 41.2%
Mejora vs baseline:   2.0%


In [8]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, log_loss, classification_report
from sklearn.calibration import CalibratedClassifierCV
import numpy as np

# Calcular pesos por clase para balancear
from sklearn.utils.class_weight import compute_sample_weight
sample_weights = compute_sample_weight('balanced', y_train)

# XGBoost con hiperparámetros ajustados
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=3,           # menos profundidad → menos overfitting
    learning_rate=0.02,    # más lento pero más preciso
    subsample=0.7,
    colsample_bytree=0.7,
    min_child_weight=5,    # evita splits con pocos ejemplos
    gamma=1,               # regularización adicional
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

xgb.fit(X_train, y_train, sample_weight=sample_weights)

# Calibrar probabilidades
model_cal = CalibratedClassifierCV(xgb, cv=5, method='isotonic')
model_cal.fit(X_train, y_train, sample_weight=sample_weights)

# Evaluar
y_pred       = model_cal.predict(X_test)
y_pred_proba = model_cal.predict_proba(X_test)

accuracy = accuracy_score(y_test, y_pred)
logloss  = log_loss(y_test, y_pred_proba)

print(f"Accuracy:  {accuracy:.1%}")
print(f"Log Loss:  {logloss:.4f}")
print(f"Baseline:  41.2%")

print(f"\nReporte por clase:")
print(classification_report(y_test, y_pred, target_names=le.classes_))

Accuracy:  37.9%
Log Loss:  1.0874
Baseline:  41.2%

Reporte por clase:
              precision    recall  f1-score   support

           A       0.39      0.38      0.38        96
           D       0.23      0.23      0.23        84
           H       0.46      0.48      0.47       126

    accuracy                           0.38       306
   macro avg       0.36      0.36      0.36       306
weighted avg       0.38      0.38      0.38       306



In [9]:
def get_team_stats_v2(team, date, df, n_recent=5):
    """Stats combinadas: forma reciente + temporada actual"""
    
    # Todos los partidos anteriores a la fecha
    home_all = df[(df['HomeTeam'] == team) & (df['Date'] < date)]
    away_all = df[(df['AwayTeam'] == team) & (df['Date'] < date)]
    all_games = pd.concat([home_all, away_all]).sort_values('Date')
    
    if len(all_games) < 3:
        return None
    
    # Temporada actual
    current_season = df[df['Date'] < date]['season'].iloc[-1]
    home_szn = home_all[home_all['season'] == current_season]
    away_szn = away_all[away_all['season'] == current_season]
    season_games = pd.concat([home_szn, away_szn]).sort_values('Date')
    
    # Últimos n partidos
    recent = all_games.tail(n_recent)
    
    def calc_stats(games, team):
        if len(games) == 0:
            return None
        gf=gc=sot_f=sot_c=wins=draws=home_wins=home_games=0
        for _, r in games.iterrows():
            is_home = r['HomeTeam'] == team
            gf    += r['FTHG'] if is_home else r['FTAG']
            gc    += r['FTAG'] if is_home else r['FTHG']
            sot_f += r['HST']  if is_home else r['AST']
            sot_c += r['AST']  if is_home else r['HST']
            won = (is_home and r['FTR']=='H') or (not is_home and r['FTR']=='A')
            drew = r['FTR'] == 'D'
            if won:  wins  += 1
            if drew: draws += 1
            if is_home:
                home_games += 1
                if r['FTR'] == 'H': home_wins += 1
        m = len(games)
        return {
            'gf_pg':       gf/m,
            'gc_pg':       gc/m,
            'dif_goles':   (gf-gc)/m,
            'sot_f_pg':    sot_f/m,
            'sot_c_pg':    sot_c/m,
            'win_rate':    wins/m,
            'draw_rate':   draws/m,
            'home_wr':     home_wins/home_games if home_games > 0 else wins/m,
        }
    
    recent_s = calc_stats(recent, team)
    season_s = calc_stats(season_games, team) if len(season_games) >= 3 else recent_s
    
    if recent_s is None:
        return None
    
    # Combinar: 60% forma reciente + 40% temporada
    combined = {}
    for k in recent_s:
        combined[f'recent_{k}'] = recent_s[k]
        combined[f'season_{k}'] = season_s[k] if season_s else recent_s[k]
    
    return combined

# Test
test = get_team_stats_v2('Arsenal', pd.Timestamp('2026-03-01'), df)
pd.DataFrame([test]).round(3)

,recent_gf_pg,season_gf_pg,recent_gc_pg,season_gc_pg,recent_dif_goles,season_dif_goles,recent_sot_f_pg,season_sot_f_pg,recent_sot_c_pg,season_sot_c_pg,recent_win_rate,season_win_rate,recent_draw_rate,season_draw_rate,recent_home_wr,season_home_wr
0,2.8,2.0,0.8,0.75,2.0,1.25,5.2,4.964,2.8,2.25,0.6,0.643,0.4,0.25,1.0,0.769


In [10]:
rows_v2 = []

for idx, partido in df.iterrows():
    home_team = partido['HomeTeam']
    away_team = partido['AwayTeam']
    date      = partido['Date']
    season    = partido['season']
    
    home_stats = get_team_stats_v2(home_team, date, df)
    away_stats = get_team_stats_v2(away_team, date, df)
    
    if home_stats is None or away_stats is None:
        continue
    
    row = {
        'date':      date,
        'season':    season,
        'home_team': home_team,
        'away_team': away_team,
        'resultado': partido['FTR'],
    }
    
    # Stats local con prefijo h_
    for k, v in home_stats.items():
        row[f'h_{k}'] = v
    
    # Stats visitante con prefijo a_
    for k, v in away_stats.items():
        row[f'a_{k}'] = v
    
    # Features diferenciales clave
    row['dif_recent_wr']    = home_stats['recent_win_rate']  - away_stats['recent_win_rate']
    row['dif_season_wr']    = home_stats['season_win_rate']  - away_stats['season_win_rate']
    row['dif_recent_gf']    = home_stats['recent_gf_pg']     - away_stats['recent_gf_pg']
    row['dif_recent_gc']    = home_stats['recent_gc_pg']     - away_stats['recent_gc_pg']
    row['dif_recent_dif']   = home_stats['recent_dif_goles'] - away_stats['recent_dif_goles']
    row['dif_season_dif']   = home_stats['season_dif_goles'] - away_stats['season_dif_goles']
    row['home_advantage']   = home_stats['recent_home_wr']   - away_stats['recent_win_rate']
    
    rows_v2.append(row)

df_v2 = pd.DataFrame(rows_v2)

print(f"Partidos: {len(df_v2)}")
print(f"Features: {len(df_v2.columns) - 5}")
print(f"\nTemporadas:")
print(df_v2['season'].value_counts().sort_index())

Partidos: 2153
Features: 39

Temporadas:
season
2021    348
2122    371
2223    374
2324    377
2425    377
2526    306
Name: count, dtype: int64


In [11]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, log_loss, classification_report
from sklearn.calibration import CalibratedClassifierCV

# Split temporal
train_v2 = df_v2[df_v2['season'] != '2526'].copy()
test_v2  = df_v2[df_v2['season'] == '2526'].copy()

# Features
feature_cols = [c for c in df_v2.columns 
                if c not in ['date','season','home_team','away_team','resultado']]

X_train_v2 = train_v2[feature_cols]
X_test_v2  = test_v2[feature_cols]

y_train_v2 = le.transform(train_v2['resultado'])
y_test_v2  = le.transform(test_v2['resultado'])

# Entrenar
xgb_v2 = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

xgb_v2.fit(X_train_v2, y_train_v2)

# Calibrar
model_v2 = CalibratedClassifierCV(xgb_v2, cv=5, method='isotonic')
model_v2.fit(X_train_v2, y_train_v2)

# Evaluar
y_pred_v2       = model_v2.predict(X_test_v2)
y_pred_proba_v2 = model_v2.predict_proba(X_test_v2)

accuracy_v2 = accuracy_score(y_test_v2, y_pred_v2)
logloss_v2  = log_loss(y_test_v2, y_pred_proba_v2)

print(f"Accuracy:  {accuracy_v2:.1%}")
print(f"Log Loss:  {logloss_v2:.4f}")
print(f"Baseline:  41.2%")
print(f"Mejora:    {(accuracy_v2 - 0.412):.1%}")

print(f"\nReporte por clase:")
print(classification_report(y_test_v2, y_pred_v2, target_names=le.classes_))

Accuracy:  44.8%
Log Loss:  1.0484
Baseline:  41.2%
Mejora:    3.6%

Reporte por clase:
              precision    recall  f1-score   support

           A       0.41      0.53      0.46        96
           D       0.50      0.01      0.02        84
           H       0.47      0.67      0.56       126

    accuracy                           0.45       306
   macro avg       0.46      0.41      0.35       306
weighted avg       0.46      0.45      0.38       306



In [12]:
# Análisis de probabilidades predichas
proba_df = pd.DataFrame(y_pred_proba_v2, columns=['prob_A', 'prob_D', 'prob_H'])
proba_df['pred'] = le.inverse_transform(y_pred_v2)
proba_df['real'] = le.inverse_transform(y_test_v2)
proba_df['correcto'] = proba_df['pred'] == proba_df['real']
proba_df['confianza'] = proba_df[['prob_A','prob_D','prob_H']].max(axis=1)
proba_df['home_team'] = test_v2['home_team'].values
proba_df['away_team'] = test_v2['away_team'].values

# Accuracy por nivel de confianza
print("Accuracy según confianza del modelo:")
bins = [0.33, 0.40, 0.45, 0.50, 0.55, 1.0]
labels = ['33-40%', '40-45%', '45-50%', '50-55%', '>55%']
proba_df['confianza_bin'] = pd.cut(proba_df['confianza'], bins=bins, labels=labels)

resumen = proba_df.groupby('confianza_bin', observed=True).agg(
    partidos=('correcto', 'count'),
    accuracy=('correcto', 'mean')
).round(3)
print(resumen)

print(f"\nPartidos con confianza > 50%: {(proba_df['confianza'] > 0.50).sum()}")
print(f"Accuracy en esos partidos:    {proba_df[proba_df['confianza'] > 0.50]['correcto'].mean():.1%}")

Accuracy según confianza del modelo:
               partidos  accuracy
confianza_bin                    
33-40%               24     0.417
40-45%               93     0.323
45-50%               64     0.469
50-55%               51     0.490
>55%                 74     0.568

Partidos con confianza > 50%: 125
Accuracy en esos partidos:    53.6%


In [13]:
def predecir_partido(home_team, away_team, fecha_str='2026-03-29'):
    """
    Predice probabilidades para un partido específico.
    Solo muestra confianza si supera el umbral del 55%.
    """
    fecha = pd.Timestamp(fecha_str)
    
    home_stats = get_team_stats_v2(home_team, fecha, df)
    away_stats = get_team_stats_v2(away_team, fecha, df)
    
    if home_stats is None or away_stats is None:
        print(f"Sin suficiente historia para {home_team} o {away_team}")
        return
    
    # Construir fila igual que el dataset
    row = {}
    for k, v in home_stats.items():
        row[f'h_{k}'] = v
    for k, v in away_stats.items():
        row[f'a_{k}'] = v
    
    row['dif_recent_wr']  = home_stats['recent_win_rate']  - away_stats['recent_win_rate']
    row['dif_season_wr']  = home_stats['season_win_rate']  - away_stats['season_win_rate']
    row['dif_recent_gf']  = home_stats['recent_gf_pg']     - away_stats['recent_gf_pg']
    row['dif_recent_gc']  = home_stats['recent_gc_pg']     - away_stats['recent_gc_pg']
    row['dif_recent_dif'] = home_stats['recent_dif_goles'] - away_stats['recent_dif_goles']
    row['dif_season_dif'] = home_stats['season_dif_goles'] - away_stats['season_dif_goles']
    row['home_advantage'] = home_stats['recent_home_wr']   - away_stats['recent_win_rate']
    
    X_new = pd.DataFrame([row])[feature_cols]
    proba = model_v2.predict_proba(X_new)[0]
    
    prob_A, prob_D, prob_H = proba[0], proba[1], proba[2]
    confianza = max(proba)
    pred = le.inverse_transform([np.argmax(proba)])[0]
    resultado_label = {'H': f'{home_team} gana', 
                       'D': 'Empate', 
                       'A': f'{away_team} gana'}
    
    print(f"\n{'='*45}")
    print(f"  {home_team} vs {away_team}")
    print(f"{'='*45}")
    print(f"  Local gana    {home_team:<20} {prob_H:.1%}")
    print(f"  Empate                               {prob_D:.1%}")
    print(f"  Visitante gana {away_team:<19} {prob_A:.1%}")
    print(f"{'─'*45}")
    
    if confianza >= 0.55:
        print(f"  Prediccion:  {resultado_label[pred]}")
        print(f"  Confianza:   {confianza:.1%}  ← USAR")
    else:
        print(f"  Confianza:   {confianza:.1%}  ← IGNORAR (bajo umbral)")
    print(f"{'='*45}")

# Probar con partidos reales de la próxima jornada
partidos_jornada = [
    ('Arsenal',   'Chelsea'),
    ('Man City',  'Liverpool'),
    ('Tottenham', 'Man United'),
    ('Newcastle', 'Aston Villa'),
    ('Brighton',  'Wolves'),
]

fecha_jornada = '2026-03-29'

for home, away in partidos_jornada:
    predecir_partido(home, away, fecha_jornada)


  Arsenal vs Chelsea
  Local gana    Arsenal              59.5%
  Empate                               9.1%
  Visitante gana Chelsea             31.4%
─────────────────────────────────────────────
  Prediccion:  Arsenal gana
  Confianza:   59.5%  ← USAR

  Man City vs Liverpool
  Local gana    Man City             56.3%
  Empate                               21.4%
  Visitante gana Liverpool           22.3%
─────────────────────────────────────────────
  Prediccion:  Man City gana
  Confianza:   56.3%  ← USAR

  Tottenham vs Man United
  Local gana    Tottenham            29.8%
  Empate                               25.0%
  Visitante gana Man United          45.2%
─────────────────────────────────────────────
  Confianza:   45.2%  ← IGNORAR (bajo umbral)

  Newcastle vs Aston Villa
  Local gana    Newcastle            42.1%
  Empate                               20.8%
  Visitante gana Aston Villa         37.1%
─────────────────────────────────────────────
  Confianza:   42.1%  ← IGNORA

In [14]:
partidos_jornada = [
    ('West Ham',          'Wolves'),
    ('Arsenal',           'Bournemouth'),
    ('Burnley',           'Brighton'),
    ('Brentford',         'Everton'),
    ('Liverpool',         'Fulham'),
    ('Sunderland',        'Tottenham'),
    ('Nott\'m Forest',    'Aston Villa'),
    ('Crystal Palace',    'Newcastle'),
    ('Chelsea',           'Man City'),
    ('Man United',        'Leeds'),
]

fecha_jornada = '2026-04-10'

for home, away in partidos_jornada:
    predecir_partido(home, away, fecha_jornada)


print(sorted(df['HomeTeam'].unique()))


  West Ham vs Wolves
  Local gana    West Ham             47.7%
  Empate                               23.0%
  Visitante gana Wolves              29.3%
─────────────────────────────────────────────
  Confianza:   47.7%  ← IGNORAR (bajo umbral)

  Arsenal vs Bournemouth
  Local gana    Arsenal              63.5%
  Empate                               14.5%
  Visitante gana Bournemouth         22.0%
─────────────────────────────────────────────
  Prediccion:  Arsenal gana
  Confianza:   63.5%  ← USAR

  Burnley vs Brighton
  Local gana    Burnley              32.0%
  Empate                               25.9%
  Visitante gana Brighton            42.2%
─────────────────────────────────────────────
  Confianza:   42.2%  ← IGNORAR (bajo umbral)

  Brentford vs Everton
  Local gana    Brentford            52.7%
  Empate                               20.4%
  Visitante gana Everton             26.9%
─────────────────────────────────────────────
  Confianza:   52.7%  ← IGNORAR (bajo umbral)

 

In [15]:
def get_h2h_stats(home_team, away_team, date, df, n=10):
    """Historial directo entre dos equipos — últimos n enfrentamientos"""
    h2h = df[
        ((df['HomeTeam'] == home_team) & (df['AwayTeam'] == away_team)) |
        ((df['HomeTeam'] == away_team) & (df['AwayTeam'] == home_team))
    ]
    h2h = h2h[h2h['Date'] < date].tail(n)
    
    if len(h2h) < 3:
        return {'h2h_home_wr': 0.33, 'h2h_draw_rate': 0.33, 'h2h_away_wr': 0.33}
    
    home_wins = ((h2h['HomeTeam'] == home_team) & (h2h['FTR'] == 'H')).sum() + \
                ((h2h['AwayTeam'] == home_team) & (h2h['FTR'] == 'A')).sum()
    draws     = (h2h['FTR'] == 'D').sum()
    away_wins = len(h2h) - home_wins - draws
    m = len(h2h)
    
    return {
        'h2h_home_wr':   home_wins / m,
        'h2h_draw_rate': draws / m,
        'h2h_away_wr':   away_wins / m,
        'h2h_partidos':  m
    }

def get_tabla_posicion(team, date, df):
    """Posición y puntos en la tabla al momento del partido"""
    season = df[df['Date'] < date]['season'].iloc[-1]
    partidos_szn = df[
        (df['season'] == season) & 
        (df['Date'] < date)
    ]
    
    equipos = pd.concat([
        partidos_szn['HomeTeam'], 
        partidos_szn['AwayTeam']
    ]).unique()
    
    tabla = []
    for eq in equipos:
        home_p = partidos_szn[partidos_szn['HomeTeam'] == eq]
        away_p = partidos_szn[partidos_szn['AwayTeam'] == eq]
        
        pts  = ((home_p['FTR']=='H').sum() * 3 + (home_p['FTR']=='D').sum() +
                (away_p['FTR']=='A').sum() * 3 + (away_p['FTR']=='D').sum())
        gf   = home_p['FTHG'].sum() + away_p['FTAG'].sum()
        gc   = home_p['FTAG'].sum() + away_p['FTHG'].sum()
        pj   = len(home_p) + len(away_p)
        
        tabla.append({'equipo': eq, 'pts': pts, 'gf': gf, 'gc': gc, 'pj': pj})
    
    tabla_df = pd.DataFrame(tabla).sort_values('pts', ascending=False).reset_index(drop=True)
    tabla_df['posicion'] = tabla_df.index + 1
    tabla_df['total_equipos'] = len(tabla_df)
    
    fila = tabla_df[tabla_df['equipo'] == team]
    if len(fila) == 0:
        return {'posicion': 10, 'pts_pg': 1.0, 'dif_goles_szn': 0, 'pct_posicion': 0.5}
    
    fila = fila.iloc[0]
    pj = max(fila['pj'], 1)
    return {
        'posicion':      fila['posicion'],
        'pts_pg':        fila['pts'] / pj,
        'dif_goles_szn': (fila['gf'] - fila['gc']) / pj,
        'pct_posicion':  1 - (fila['posicion'] / fila['total_equipos'])
    }

# Test
print("H2H Arsenal vs Man City:")
print(get_h2h_stats('Arsenal', 'Man City', pd.Timestamp('2026-03-01'), df))

print("\nTabla Arsenal:")
print(get_tabla_posicion('Arsenal', pd.Timestamp('2026-03-01'), df))

H2H Arsenal vs Man City:
{'h2h_home_wr': np.float64(0.2), 'h2h_draw_rate': np.float64(0.3), 'h2h_away_wr': np.float64(0.5), 'h2h_partidos': 10}

Tabla Arsenal:
{'posicion': np.int64(1), 'pts_pg': np.float64(2.1785714285714284), 'dif_goles_szn': np.float64(1.25), 'pct_posicion': np.float64(0.95)}


In [16]:
# 1. Redefinir la función sin h2h_partidos
def get_h2h_stats(home_team, away_team, date, df, n=10):
    h2h = df[
        ((df['HomeTeam'] == home_team) & (df['AwayTeam'] == away_team)) |
        ((df['HomeTeam'] == away_team) & (df['AwayTeam'] == home_team))
    ]
    h2h = h2h[h2h['Date'] < date].tail(n)
    
    if len(h2h) < 3:
        return {'h2h_home_wr': 0.33, 'h2h_draw_rate': 0.33, 'h2h_away_wr': 0.33}
    
    home_wins = ((h2h['HomeTeam'] == home_team) & (h2h['FTR'] == 'H')).sum() + \
                ((h2h['AwayTeam'] == home_team) & (h2h['FTR'] == 'A')).sum()
    draws     = (h2h['FTR'] == 'D').sum()
    away_wins = len(h2h) - home_wins - draws
    m = len(h2h)
    
    return {
        'h2h_home_wr':   home_wins / m,
        'h2h_draw_rate': draws / m,
        'h2h_away_wr':   away_wins / m
    }

print("Función corregida")

Función corregida


In [17]:
rows_v3 = []

for idx, partido in df.iterrows():
    home_team = partido['HomeTeam']
    away_team = partido['AwayTeam']
    date      = partido['Date']
    season    = partido['season']

    home_stats = get_team_stats_v2(home_team, date, df)
    away_stats = get_team_stats_v2(away_team, date, df)

    if home_stats is None or away_stats is None:
        continue

    # H2H y tabla
    h2h        = get_h2h_stats(home_team, away_team, date, df)
    home_tabla = get_tabla_posicion(home_team, date, df)
    away_tabla = get_tabla_posicion(away_team, date, df)

    row = {
        'date':      date,
        'season':    season,
        'home_team': home_team,
        'away_team': away_team,
        'resultado': partido['FTR'],
    }

    for k, v in home_stats.items():
        row[f'h_{k}'] = v
    for k, v in away_stats.items():
        row[f'a_{k}'] = v

    # H2H
    for k, v in h2h.items():
        row[k] = v

    # Tabla local
    for k, v in home_tabla.items():
        row[f'h_tabla_{k}'] = v

    # Tabla visitante
    for k, v in away_tabla.items():
        row[f'a_tabla_{k}'] = v

    # Features diferenciales
    row['dif_recent_wr']   = home_stats['recent_win_rate']  - away_stats['recent_win_rate']
    row['dif_season_wr']   = home_stats['season_win_rate']  - away_stats['season_win_rate']
    row['dif_recent_gf']   = home_stats['recent_gf_pg']     - away_stats['recent_gf_pg']
    row['dif_recent_gc']   = home_stats['recent_gc_pg']     - away_stats['recent_gc_pg']
    row['dif_recent_dif']  = home_stats['recent_dif_goles'] - away_stats['recent_dif_goles']
    row['dif_season_dif']  = home_stats['season_dif_goles'] - away_stats['season_dif_goles']
    row['home_advantage']  = home_stats['recent_home_wr']   - away_stats['recent_win_rate']
    row['dif_pts_pg']      = home_tabla['pts_pg']           - away_tabla['pts_pg']
    row['dif_posicion']    = away_tabla['posicion']         - home_tabla['posicion']

    rows_v3.append(row)

df_v3 = pd.DataFrame(rows_v3)

print(f"Partidos: {len(df_v3)}")
print(f"Features: {len(df_v3.columns) - 5}")

Partidos: 2153
Features: 52


In [18]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, log_loss, classification_report
from sklearn.calibration import CalibratedClassifierCV

# Split temporal
train_v3 = df_v3[df_v3['season'] != '2526'].copy()
test_v3  = df_v3[df_v3['season'] == '2526'].copy()

# Features
feature_cols_v3 = [c for c in df_v3.columns 
                   if c not in ['date','season','home_team','away_team','resultado']]

X_train_v3 = train_v3[feature_cols_v3]
X_test_v3  = test_v3[feature_cols_v3]

y_train_v3 = le.transform(train_v3['resultado'])
y_test_v3  = le.transform(test_v3['resultado'])

# Entrenar
xgb_v3 = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

xgb_v3.fit(X_train_v3, y_train_v3)

# Calibrar
model_v3 = CalibratedClassifierCV(xgb_v3, cv=5, method='isotonic')
model_v3.fit(X_train_v3, y_train_v3)

# Evaluar
y_pred_v3       = model_v3.predict(X_test_v3)
y_pred_proba_v3 = model_v3.predict_proba(X_test_v3)

accuracy_v3 = accuracy_score(y_test_v3, y_pred_v3)
logloss_v3  = log_loss(y_test_v3, y_pred_proba_v3)

print(f"Accuracy v1 (original):  44.8%")
print(f"Accuracy v3 (H2H+tabla): {accuracy_v3:.1%}")
print(f"Log Loss:                {logloss_v3:.4f}")
print(f"Baseline:                41.2%")
print(f"Mejora vs baseline:      {(accuracy_v3 - 0.412):.1%}")

print(f"\nReporte por clase:")
print(classification_report(y_test_v3, y_pred_v3, target_names=le.classes_))

# Accuracy por confianza
proba_df_v3 = pd.DataFrame(y_pred_proba_v3, columns=['prob_A','prob_D','prob_H'])
proba_df_v3['confianza'] = proba_df_v3[['prob_A','prob_D','prob_H']].max(axis=1)
proba_df_v3['pred']      = le.inverse_transform(y_pred_v3)
proba_df_v3['real']      = le.inverse_transform(y_test_v3)
proba_df_v3['correcto']  = proba_df_v3['pred'] == proba_df_v3['real']

print(f"\nAccuracy por confianza:")
bins   = [0.33, 0.40, 0.45, 0.50, 0.55, 1.0]
labels = ['33-40%','40-45%','45-50%','50-55%','>55%']
proba_df_v3['confianza_bin'] = pd.cut(proba_df_v3['confianza'], bins=bins, labels=labels)
print(proba_df_v3.groupby('confianza_bin', observed=True).agg(
    partidos=('correcto','count'),
    accuracy=('correcto','mean')
).round(3))

print(f"\nPartidos con confianza >55%: {(proba_df_v3['confianza']>0.55).sum()}")
print(f"Accuracy en esos partidos:   {proba_df_v3[proba_df_v3['confianza']>0.55]['correcto'].mean():.1%}")

Accuracy v1 (original):  44.8%
Accuracy v3 (H2H+tabla): 48.4%
Log Loss:                1.0337
Baseline:                41.2%
Mejora vs baseline:      7.2%

Reporte por clase:
              precision    recall  f1-score   support

           A       0.43      0.57      0.49        96
           D       1.00      0.02      0.05        84
           H       0.51      0.72      0.60       126

    accuracy                           0.48       306
   macro avg       0.65      0.44      0.38       306
weighted avg       0.62      0.48      0.41       306


Accuracy por confianza:
               partidos  accuracy
confianza_bin                    
33-40%               32     0.438
40-45%               79     0.418
45-50%               61     0.426
50-55%               41     0.512
>55%                 93     0.581

Partidos con confianza >55%: 93
Accuracy en esos partidos:   58.1%


In [19]:
def predecir_partido_v3(home_team, away_team, fecha_str):
    fecha = pd.Timestamp(fecha_str)
    
    home_stats = get_team_stats_v2(home_team, fecha, df)
    away_stats = get_team_stats_v2(away_team, fecha, df)
    h2h        = get_h2h_stats(home_team, away_team, fecha, df)
    home_tabla = get_tabla_posicion(home_team, fecha, df)
    away_tabla = get_tabla_posicion(away_team, fecha, df)
    
    if home_stats is None or away_stats is None:
        print(f"Sin suficiente historia para {home_team} o {away_team}")
        return
    
    row = {}
    for k, v in home_stats.items(): row[f'h_{k}'] = v
    for k, v in away_stats.items(): row[f'a_{k}'] = v
    for k, v in h2h.items():        row[k] = v
    for k, v in home_tabla.items(): row[f'h_tabla_{k}'] = v
    for k, v in away_tabla.items(): row[f'a_tabla_{k}'] = v
    
    row['dif_recent_wr']  = home_stats['recent_win_rate']  - away_stats['recent_win_rate']
    row['dif_season_wr']  = home_stats['season_win_rate']  - away_stats['season_win_rate']
    row['dif_recent_gf']  = home_stats['recent_gf_pg']     - away_stats['recent_gf_pg']
    row['dif_recent_gc']  = home_stats['recent_gc_pg']     - away_stats['recent_gc_pg']
    row['dif_recent_dif'] = home_stats['recent_dif_goles'] - away_stats['recent_dif_goles']
    row['dif_season_dif'] = home_stats['season_dif_goles'] - away_stats['season_dif_goles']
    row['home_advantage'] = home_stats['recent_home_wr']   - away_stats['recent_win_rate']
    row['dif_pts_pg']     = home_tabla['pts_pg']           - away_tabla['pts_pg']
    row['dif_posicion']   = away_tabla['posicion']         - home_tabla['posicion']
    
    X_new  = pd.DataFrame([row])[feature_cols_v3]
    proba  = model_v3.predict_proba(X_new)[0]
    
    prob_A, prob_D, prob_H = proba[0], proba[1], proba[2]
    confianza = max(proba)
    pred = le.inverse_transform([np.argmax(proba)])[0]
    
    resultado_label = {
        'H': f'{home_team} gana',
        'D': 'Empate',
        'A': f'{away_team} gana'
    }
    
    usar = confianza >= 0.55
    
    print(f"\n{'='*48}")
    print(f"  {home_team} vs {away_team}")
    print(f"{'='*48}")
    print(f"  Local gana     {home_team:<22} {prob_H:.1%}")
    print(f"  Empate                                  {prob_D:.1%}")
    print(f"  Visitante gana {away_team:<22} {prob_A:.1%}")
    print(f"{'─'*48}")
    print(f"  Prediccion:  {resultado_label[pred]}")
    print(f"  Confianza:   {confianza:.1%}  {'← USAR' if usar else '← IGNORAR'}")
    print(f"{'='*48}")

# Jornada del 10-13 Abril 2026
partidos_jornada = [
    ('West Ham',       'Wolves'),
    ('Arsenal',        'Bournemouth'),
    ('Burnley',        'Brighton'),
    ('Brentford',      'Everton'),
    ('Liverpool',      'Fulham'),
    ('Sunderland',     'Tottenham'),
    ('Nott\'m Forest', 'Aston Villa'),
    ('Crystal Palace', 'Newcastle'),
    ('Chelsea',        'Man City'),
    ('Man United',     'Leeds'),
]

fecha_jornada = '2026-04-10'

# Verificar nombres exactos primero
print("Equipos disponibles en el dataset:")
print(sorted(df['HomeTeam'].unique()))

Equipos disponibles en el dataset:
['Arsenal', 'Aston Villa', 'Bournemouth', 'Brentford', 'Brighton', 'Burnley', 'Chelsea', 'Crystal Palace', 'Everton', 'Fulham', 'Ipswich', 'Leeds', 'Leicester', 'Liverpool', 'Luton', 'Man City', 'Man United', 'Newcastle', 'Norwich', "Nott'm Forest", 'Sheffield United', 'Southampton', 'Sunderland', 'Tottenham', 'Watford', 'West Brom', 'West Ham', 'Wolves']


In [20]:
partidos_jornada = [
    ('West Ham',       'Wolves'),
    ('Arsenal',        'Bournemouth'),
    ('Burnley',        'Brighton'),
    ('Brentford',      'Everton'),
    ('Liverpool',      'Fulham'),
    ('Sunderland',     'Tottenham'),
    ("Nott'm Forest",  'Aston Villa'),
    ('Crystal Palace', 'Newcastle'),
    ('Chelsea',        'Man City'),
    ('Man United',     'Leeds'),
]

fecha_jornada = '2026-04-10'

for home, away in partidos_jornada:
    predecir_partido_v3(home, away, fecha_jornada)


  West Ham vs Wolves
  Local gana     West Ham               55.2%
  Empate                                  23.2%
  Visitante gana Wolves                 21.6%
────────────────────────────────────────────────
  Prediccion:  West Ham gana
  Confianza:   55.2%  ← USAR

  Arsenal vs Bournemouth
  Local gana     Arsenal                64.8%
  Empate                                  20.4%
  Visitante gana Bournemouth            14.8%
────────────────────────────────────────────────
  Prediccion:  Arsenal gana
  Confianza:   64.8%  ← USAR

  Burnley vs Brighton
  Local gana     Burnley                33.2%
  Empate                                  27.5%
  Visitante gana Brighton               39.3%
────────────────────────────────────────────────
  Prediccion:  Brighton gana
  Confianza:   39.3%  ← IGNORAR

  Brentford vs Everton
  Local gana     Brentford              47.8%
  Empate                                  22.6%
  Visitante gana Everton                29.6%
──────────────────────

In [21]:
# ============================================
# SEGUIMIENTO SEMANAL DE PREDICCIONES
# ============================================

import pandas as pd
from datetime import datetime

# Base de datos de predicciones
predicciones = []

def registrar_prediccion(jornada, fecha, home, away, pred, prob_H, prob_D, prob_A, confianza):
    """Registra una predicción antes del partido"""
    predicciones.append({
        'jornada':    jornada,
        'fecha':      fecha,
        'home':       home,
        'away':       away,
        'prediccion': pred,
        'prob_H':     prob_H,
        'prob_D':     prob_D,
        'prob_A':     prob_A,
        'confianza':  confianza,
        'resultado':  None,
        'correcto':   None
    })
    print(f"Registrado: {home} vs {away} → {pred} ({confianza:.1%})")

def registrar_resultado(home, away, resultado_real):
    """
    Registra el resultado real después del partido.
    resultado_real: 'H', 'D', o 'A'
    """
    for p in predicciones:
        if p['home'] == home and p['away'] == away:
            p['resultado'] = resultado_real
            p['correcto']  = p['prediccion'] == resultado_real
            estado = "ACERTADO" if p['correcto'] else "FALLADO"
            print(f"{estado}: {home} vs {away} | Pred: {p['prediccion']} | Real: {resultado_real}")
            return
    print(f"Partido no encontrado: {home} vs {away}")

def ver_seguimiento():
    """Muestra el resumen completo de predicciones"""
    if not predicciones:
        print("Sin predicciones registradas aún")
        return
    
    df_seg = pd.DataFrame(predicciones)
    
    # Resumen general
    total      = len(df_seg)
    jugados    = df_seg['resultado'].notna().sum()
    acertados  = df_seg['correcto'].sum() if jugados > 0 else 0
    accuracy   = acertados / jugados if jugados > 0 else 0
    
    print(f"\n{'='*50}")
    print(f"  RESUMEN GENERAL")
    print(f"{'='*50}")
    print(f"  Total predicciones:  {total}")
    print(f"  Partidos jugados:    {jugados}")
    print(f"  Acertados:           {int(acertados)}")
    print(f"  Accuracy real:       {accuracy:.1%}")
    print(f"  Baseline (siempre H):{0.412:.1%}")
    print(f"  Diferencia:          {(accuracy - 0.412):+.1%}")
    
    # Detalle por jornada
    print(f"\n{'='*50}")
    print(f"  DETALLE POR PARTIDO")
    print(f"{'='*50}")
    for _, row in df_seg.iterrows():
        if row['resultado'] is not None:
            icon = "✓" if row['correcto'] else "✗"
            print(f"  {icon} J{row['jornada']} {row['home']} vs {row['away']}")
            print(f"    Pred: {row['prediccion']} ({row['confianza']:.1%}) | Real: {row['resultado']}")
        else:
            print(f"  ? J{row['jornada']} {row['home']} vs {row['away']}")
            print(f"    Pred: {row['prediccion']} ({row['confianza']:.1%}) | Pendiente")
    
    # Accuracy por confianza
    if jugados > 0:
        print(f"\n{'='*50}")
        print(f"  ACCURACY POR CONFIANZA")
        print(f"{'='*50}")
        df_jug = df_seg[df_seg['resultado'].notna()].copy()
        bins   = [0.50, 0.55, 0.60, 0.65, 1.0]
        labels = ['50-55%', '55-60%', '60-65%', '>65%']
        df_jug['conf_bin'] = pd.cut(df_jug['confianza'], bins=bins, labels=labels)
        resumen = df_jug.groupby('conf_bin', observed=True).agg(
            partidos=('correcto', 'count'),
            acertados=('correcto', 'sum'),
            accuracy=('correcto', 'mean')
        ).round(3)
        print(resumen.to_string())

print("Celdas de seguimiento listas")

Celdas de seguimiento listas


In [22]:
# ============================================
# REGISTRAR RESULTADOS — después de los partidos
# Cambia 'H', 'D', o 'A' según el resultado real
# ============================================

# registrar_resultado('West Ham',  'Wolves',      'H')  # ← cambiar después del partido
# registrar_resultado('Arsenal',   'Bournemouth', 'H')  # ← cambiar después del partido

# Ver resumen actualizado
ver_seguimiento()

Sin predicciones registradas aún


In [23]:
import json

def guardar_predicciones(archivo='predicciones.json'):
    """Guarda predicciones en disco para no perderlas al reiniciar"""
    datos = []
    for p in predicciones:
        p_copy = p.copy()
        p_copy['correcto'] = bool(p['correcto']) if p['correcto'] is not None else None
        datos.append(p_copy)
    with open(archivo, 'w') as f:
        json.dump(datos, f, indent=2)
    print(f"Guardado: {len(datos)} predicciones en {archivo}")

def cargar_predicciones(archivo='predicciones.json'):
    """Carga predicciones desde disco al reiniciar el kernel"""
    global predicciones
    try:
        with open(archivo, 'r') as f:
            predicciones = json.load(f)
        print(f"Cargado: {len(predicciones)} predicciones desde {archivo}")
    except FileNotFoundError:
        print("No hay predicciones guardadas aún")

# Guardar después de registrar cada predicción o resultado
guardar_predicciones()

Guardado: 0 predicciones en predicciones.json


In [24]:
import pickle

# Guardar modelo y todo lo necesario
with open('modelo_premier.pkl', 'wb') as f:
    pickle.dump({
        'model_v3':       model_v3,
        'feature_cols_v3': feature_cols_v3,
        'le':             le,
        'df':             df
    }, f)

print("Modelo guardado en modelo_premier.pkl")

Modelo guardado en modelo_premier.pkl


In [25]:
# Hoy
# └── Correr notebook completo
# └── guardar_predicciones()
# └── pickle.dump(modelo)

# Viernes / Sábado
# └── Abrir notebook
# └── Correr solo imports (pandas, numpy, etc.)
# └── Cargar pickle  ←  evita reconstruir todo
# └── cargar_predicciones()
# └── registrar_resultado(...)
# └── ver_seguimiento()
# └── guardar_predicciones()

import pickle

with open('modelo_premier.pkl', 'rb') as f:
    datos = pickle.load(f)

model_v3        = datos['model_v3']
feature_cols_v3 = datos['feature_cols_v3']
le              = datos['le']
df              = datos['df']

print("Modelo cargado correctamente")

Modelo cargado correctamente


# Desde aca funciona nuestro modelo

- Hoy          →  Celdas 1, 2, 3
- Viernes 10   →  Celdas 4, 5, 6, 7 → actualizar celda 8 (West Ham) → celda 9
- Sábado 11    →  Celdas 4, 5, 6, 7 → actualizar celda 8 (Arsenal)  → celda 9

In [26]:
# ============================================
# CELDA 1 — CORRER HOY
# Guardar modelo en disco después de entrenar
# ============================================
import pickle

with open('modelo_premier.pkl', 'wb') as f:
    pickle.dump({
        'model_v3':        model_v3,
        'feature_cols_v3': feature_cols_v3,
        'le':              le,
        'df':              df
    }, f)

print("Modelo guardado correctamente")

Modelo guardado correctamente


In [27]:
# ============================================
# CELDA 2 — CORRER HOY
# Registrar predicciones jornada 32
# ============================================

registrar_prediccion(
    jornada=32, fecha='2026-04-10',
    home='West Ham', away='Wolves',
    pred='H', prob_H=0.552, prob_D=0.232, prob_A=0.216,
    confianza=0.552
)

registrar_prediccion(
    jornada=32, fecha='2026-04-11',
    home='Arsenal', away='Bournemouth',
    pred='H', prob_H=0.648, prob_D=0.204, prob_A=0.148,
    confianza=0.648
)

print("\nPredicciones registradas:")
ver_seguimiento()

Registrado: West Ham vs Wolves → H (55.2%)
Registrado: Arsenal vs Bournemouth → H (64.8%)

Predicciones registradas:

  RESUMEN GENERAL
  Total predicciones:  2
  Partidos jugados:    0
  Acertados:           0
  Accuracy real:       0.0%
  Baseline (siempre H):41.2%
  Diferencia:          -41.2%

  DETALLE POR PARTIDO
  ? J32 West Ham vs Wolves
    Pred: H (55.2%) | Pendiente
  ? J32 Arsenal vs Bournemouth
    Pred: H (64.8%) | Pendiente


In [28]:
# ============================================
# CELDA 3 — CORRER HOY
# Guardar predicciones en disco
# ============================================

guardar_predicciones()
print("Listo — cierra el notebook")

Guardado: 2 predicciones en predicciones.json
Listo — cierra el notebook


In [29]:
# # ============================================
# # CELDA 4 — CORRER EL VIERNES / SÁBADO
# # Imports básicos
# # ============================================

# import pickle
# import pandas as pd
# import numpy as np
# import json

In [30]:
# # ============================================
# # CELDA 5 — CORRER EL VIERNES / SÁBADO
# # Cargar modelo desde disco
# # ============================================

# with open('modelo_premier.pkl', 'rb') as f:
#     datos = pickle.load(f)

# model_v3        = datos['model_v3']
# feature_cols_v3 = datos['feature_cols_v3']
# le              = datos['le']
# df              = datos['df']

# print("Modelo cargado correctamente")

In [31]:
# # ============================================
# # CELDA 6 — CORRER EL VIERNES / SÁBADO
# # Cargar funciones de seguimiento
# # ============================================

# predicciones = []

# def registrar_prediccion(jornada, fecha, home, away, pred, prob_H, prob_D, prob_A, confianza):
#     predicciones.append({
#         'jornada': jornada, 'fecha': fecha,
#         'home': home, 'away': away,
#         'prediccion': pred, 'prob_H': prob_H,
#         'prob_D': prob_D, 'prob_A': prob_A,
#         'confianza': confianza, 'resultado': None, 'correcto': None
#     })
#     print(f"Registrado: {home} vs {away} → {pred} ({confianza:.1%})")

# def registrar_resultado(home, away, resultado_real):
#     for p in predicciones:
#         if p['home'] == home and p['away'] == away:
#             p['resultado'] = resultado_real
#             p['correcto']  = p['prediccion'] == resultado_real
#             estado = "ACERTADO" if p['correcto'] else "FALLADO"
#             print(f"{estado}: {home} vs {away} | Pred: {p['prediccion']} | Real: {resultado_real}")
#             return
#     print(f"Partido no encontrado: {home} vs {away}")

# def ver_seguimiento():
#     if not predicciones:
#         print("Sin predicciones registradas aún")
#         return
#     df_seg    = pd.DataFrame(predicciones)
#     total     = len(df_seg)
#     jugados   = df_seg['resultado'].notna().sum()
#     acertados = df_seg['correcto'].sum() if jugados > 0 else 0
#     accuracy  = acertados / jugados if jugados > 0 else 0
#     print(f"\n{'='*50}")
#     print(f"  RESUMEN GENERAL")
#     print(f"{'='*50}")
#     print(f"  Total predicciones:  {total}")
#     print(f"  Partidos jugados:    {jugados}")
#     print(f"  Acertados:           {int(acertados)}")
#     print(f"  Accuracy real:       {accuracy:.1%}")
#     print(f"  Baseline:            41.2%")
#     print(f"  Diferencia:          {(accuracy - 0.412):+.1%}")
#     print(f"\n{'='*50}")
#     print(f"  DETALLE POR PARTIDO")
#     print(f"{'='*50}")
#     for _, row in df_seg.iterrows():
#         if row['resultado'] is not None:
#             icon = "✓" if row['correcto'] else "✗"
#             print(f"  {icon} J{row['jornada']} {row['home']} vs {row['away']}")
#             print(f"    Pred: {row['prediccion']} ({row['confianza']:.1%}) | Real: {row['resultado']}")
#         else:
#             print(f"  ? J{row['jornada']} {row['home']} vs {row['away']}")
#             print(f"    Pred: {row['prediccion']} ({row['confianza']:.1%}) | Pendiente")

# def guardar_predicciones(archivo='predicciones.json'):
#     datos = []
#     for p in predicciones:
#         p_copy = p.copy()
#         p_copy['correcto'] = bool(p['correcto']) if p['correcto'] is not None else None
#         datos.append(p_copy)
#     with open(archivo, 'w') as f:
#         json.dump(datos, f, indent=2)
#     print(f"Guardado: {len(datos)} predicciones")

# def cargar_predicciones(archivo='predicciones.json'):
#     global predicciones
#     try:
#         with open(archivo, 'r') as f:
#             predicciones = json.load(f)
#         print(f"Cargado: {len(predicciones)} predicciones")
#     except FileNotFoundError:
#         print("No hay predicciones guardadas aún")

# print("Funciones cargadas correctamente")

In [32]:
# # ============================================
# # CELDA 7 — CORRER EL VIERNES / SÁBADO
# # Cargar predicciones guardadas
# # ============================================

# cargar_predicciones()
# ver_seguimiento()

In [33]:
# # ============================================
# # CELDA 8 — CORRER DESPUÉS DE CADA PARTIDO
# # Cambiar 'H' / 'D' / 'A' por el resultado real
# #
# # H = gana el local
# # D = empate
# # A = gana el visitante
# # ============================================

# # Viernes 10 abril
# registrar_resultado('West Ham',  'Wolves',      'H')  # ← cambiar por resultado real

# # Sábado 11 abril — correr después del partido
# registrar_resultado('Arsenal',   'Bournemouth', 'H')  # ← cambiar por resultado real

In [34]:
# # ============================================
# # CELDA 9 — CORRER DESPUÉS DE CADA RESULTADO
# # Ver resumen y guardar
# # ============================================

# ver_seguimiento()
# guardar_predicciones()